In [1]:
from openai import OpenAI
from dotenv import load_dotenv
from glob import glob
import os
import json
import base64

load_dotenv()
strAPI_key = os.getenv("OPENAI_API_KEY")    # Load OpenAI API Key
client = OpenAI(api_key=strAPI_key)         # Create OpenAI Client Instance 

def encode_image(image_path):               # Encode image to base64 string
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

In [4]:
def image_quiz(image_path, n_trial=0, max_trial=3):
    if n_trial >= max_trial:                # If Max trial exceeded Then raise Exception
        raise Exception("Failed to generate a quiz.")
        
    base64_image = encode_image(image_path) # Encode image to base64 string

    quiz_prompt = """
    제공된 이미지를 바탕으로, 다음과 같은 형식의 퀴즈를 만들어주세요. 
    정답은 1~4 중 하나만 해당하도록 출제하세요.
    토익 리스닝 문제 스타일로 문제를 만들어주세요.
    아래는 예시입니다. 
    ----- 예시 -----

    [Quiz]: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
    - (1) 베이커리에서 사람들이 빵을 사고 있는 모습이 담겨 있습니다.
    - (2) 맨 앞에 서 있는 사람은 빨간색 셔츠를 입고 있습니다.
    - (3) 기차를 타기 위해 줄을 서 있는 사람들이 있습니다.
    - (4) 점원은 노란색 티셔츠를 입고 있습니다.
    - (5) 답없음.
        
    [Answer]: (4) 점원은 노란색 티셔츠가 아닌 파란색 티셔츠를 입고 있습니다.
    
    ======
    """
    strMessages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": quiz_prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}",
                        },
                    },
                ],
            }
        ]

    response = client.chat.completions.create(
            model="gpt-4o",         # Specify GPT-4o model
            messages = strMessages  # Sent strMessages to messages parameter
    )
        
    return response.choices[0].message.content

In [5]:
Quiz = image_quiz("../data/Image_Sample.jpg")
print(Quiz)

[Quiz]: 다음 이미지에 대한 설명 중 옳지 않은 것은 무엇인가요?
- (1) 배 위에 사람이 타고 있습니다.
- (2) 여러 마리의 새들이 날고 있습니다.
- (3) 바다는 아주 고요해 보입니다.
- (4) 이미지는 컬러로 표현되어 있습니다.
- (5) 답없음.

[Answer]: (4) 이미지는 흑백으로 표현되어 있습니다.
